## Packages Import

In [39]:
import requests
import os
import yaml
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import pandas as pd

# Apollo Scraper

In [40]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
auth = HTTPBasicAuth(username, password)

In [41]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"
response = requests.get(url, auth=auth)
response.encoding = 'utf-8' 
print(f"Status code: {response.status_code}")

Status code: 200


In [42]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [43]:
group_tag = page_dom.select_one("div.grupa")
group = group_tag.get_text().strip() if group_tag else "Not found"
print(f"Group: {group}")

Group: ZICSS1-1212


In [44]:
classes_tag = page_dom.select_one("table")

if classes_tag:
    with open("temp.html", "w", encoding="utf-8") as hf:
        hf.write(str(classes_tag))
    classes = pd.read_html("temp.html", encoding="utf-8")[0]
    os.remove("temp.html")
    print("Table loaded successfully.")
else:
    print("Table not found!")

Table loaded successfully.


In [45]:
classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"]), "Typ"] = classes['Typ'].str.capitalize()
print(classes.columns.tolist())

['Termin', 'Dzień, godzina', 'Przedmiot', 'Typ', 'Nauczyciel', 'Sala']


In [46]:
classes[['Day','Start time', 'hyphen', 'End time', "Duration"]] = classes.iloc[:, 1].str.split(' ', expand=True)

In [47]:
classes['Duration'] = classes['Duration'].str.extract(r'\((.*?)g')

In [48]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [49]:
if not os.path.exists("./schedules"):
    os.mkdir("./schedules")

In [50]:
classes.to_csv(f"schedules/{group}.csv", encoding="UTF-8-sig", index=False)
print(f"Successfully saved to schedules/{group}.csv")

Successfully saved to schedules/ZICSS1-1212.csv
